# **TP1 — Simulación del embarque de un avión**

## Objetivo

El objetivo de este trabajo es simular el proceso de embarque de un avión y comparar distintas políticas de boarding para determinar cuál minimiza el tiempo total necesario para que todos los pasajeros estén sentados.

El avión cuenta con:
- 25 filas.
- 4 asientos por fila.
- 100 pasajeros en total.
- Un único pasillo central de ancho suficiente para una persona.

Cada pasajero tiene un asiento asignado `(fila, asiento)` y puede tener o no carry-on.


> Importamos las librerias que más usaremos en el TP.

In [1]:
import numpy as np
import matplotlib as plt
import random

**Decisiones de modelado**

La simulación avanza en intervalos de **1 segundo**.

Cada pasajero puede encontrarse en uno de los siguientes estados:

- `esperando`: todavía no ingresó al avión.
- `caminando`: se está desplazando por el pasillo.
- `guardando`: llegó a su fila y está guardando su carry-on.
- `interferencia`: debe esperar porque otro pasajero tiene que levantarse para permitirle acceder a su asiento.
- `sentandose`: está ocupando su asiento.
- `sentado`: terminó el proceso de embarque.

Los pasajeros ingresan al avión respetando un orden previamente determinado por la política de embarque.

**Parámetros**

| Acción | Tiempo |
|---|---:|
| Sentarse | 4 segundos |
| Guardar carry-on | 10 segundos |
| Interferencia entre pasajeros | 8 segundos |
| Caminar una fila sin carry-on | 3 segundos |
| Caminar una fila con carry-on | 6 segundos |

La probabilidad de que cada pasajero tenga carry-on se representa mediante `p`.

$$
P(\text{carry-on}) = p
$$

> Sumamos `json` y `pathlib`, que vamos a necesitar más adelante para guardar los HTML de las animaciones.

In [2]:
import json
from pathlib import Path

> Estos son los parámetros fijos del modelo: cuánto tarda cada acción y la semilla para que la simulación sea reproducible.

In [ ]:
# PARÁMETROS DEL MODELO

TIEMPO_SENTARSE = 4
TIEMPO_GUARDAR = 10
TIEMPO_INTERFERENCIA = 8

P_CARRY_ON = 0.5

# Semilla opcional para poder repetir exactamente la misma corrida
SEMILLA = 42
random.seed(42)

**Representación de los pasajeros**

Cada pasajero se representa mediante un conjunto de atributos que permiten seguir su evolución durante la simulación:

- **fila:** fila asignada;
- **asiento:** posición dentro de la fila;
- **carry_on:** indica si lleva equipaje de mano;
- **posición:** lugar que ocupa actualmente en el pasillo;
- **estado:** acción que está realizando;
- **tiempo_estado:** cantidad de segundos que lleva realizando esa acción.

Los posibles estados son:

`esperando → caminando → guardando → interferencia → sentándose → sentado`

No todos los pasajeros atraviesan necesariamente todos los estados. Por ejemplo, un pasajero sin carry-on no pasa por el estado `guardando`, y un pasajero que no encuentra a nadie bloqueando su asiento no pasa por `interferencia`.

Esta representación permite modelar el boarding como un sistema dinámico en el que cada pasajero cambia de estado según su situación y la de los pasajeros que lo rodean.

> Creamos los 100 pasajeros del avión, cada uno con su fila, asiento y si lleva o no equipaje de mano.

In [ ]:
# CREAR PASAJEROS

def crear_pasajeros(p=P_CARRY_ON):

    pasajeros = []
    contador = 1

    for fila in range(1, 26):

        for asiento in [-2, -1, 1, 2]:

            pasajero = {
                "nombre": f"P{contador}",
                "fila": fila,
                "asiento": asiento,
                "carry_on": random.random() < p,
                "posicion": 0,
                "estado": "esperando",
                "tiempo_estado": 0,

                # Para manejar interferencias
                "interfiere_con": None,
                "bloqueando_a": None
            }

            pasajeros.append(pasajero)

            contador += 1

    return pasajeros

**Políticas de embarque**

Una vez creados los 100 pasajeros, se determina su orden de ingreso al avión según la política de embarque analizada.

Es importante notar que **las reglas físicas de movimiento son las mismas para todas las políticas**. Lo único que cambia es el orden inicial de los pasajeros. De esta manera, las diferencias observadas en los tiempos de embarque pueden atribuirse al mecanismo de ordenamiento y no a cambios en las reglas de la simulación.

> Función que reordena a los pasajeros según la política de embarque elegida (Random, Back-to-Front, WILMA o Steffen).

In [ ]:
# POLÍTICAS DE EMBARQUE

def ordenar_pasajeros(pasajeros, politica):

    # RANDOM

    if politica == "Random":

        random.shuffle(pasajeros)

    # BACK-TO-FRONT

    elif politica == "Back-to-Front":

        # Primero entran las filas del fondo.
        # Dentro de una misma fila mezclamos aleatoriamente
        # los cuatro pasajeros.

        grupos = []

        for fila in range(25, 0, -1):

            pasajeros_fila = [
                p for p in pasajeros
                if p["fila"] == fila
            ]

            random.shuffle(pasajeros_fila)

            grupos.extend(pasajeros_fila)

        pasajeros = grupos

    # WILMA

    elif politica == "WILMA":

        # Primero entran todos los pasajeros de ventana:
        # -2 y 2
        
        # Después entran los pasajeros de pasillo:
        # -1 y 1

        ventanas = [
            p for p in pasajeros
            if p["asiento"] in [-2, 2]
        ]

        pasillos = [
            p for p in pasajeros
            if p["asiento"] in [-1, 1]
        ]

        # Dentro de cada grupo usamos orden aleatorio
        random.shuffle(ventanas)
        random.shuffle(pasillos)

        pasajeros = ventanas + pasillos

    # STEFFEN

    elif politica == "Steffen":

        # Adaptación del método Steffen a la configuración 2+2
        # (ventana izquierda -2, pasillo izquierdo -1,
        # pasillo derecho 1, ventana derecha 2; no hay asiento
        # central).
        
        # 1. Ventanas (-2, 2) antes que pasillos (-1, 1).
        # 2. Dentro de cada tipo de asiento, se separan las
        #    filas impares de las pares.
        # 3. Dentro de cada grupo de filas, se recorre de atrás
        #    hacia adelante en DOS pasadas que alternan el lado
        #    (izquierda/derecha), para que dos pasajeros
        #    consecutivos nunca compartan fila.

        orden = []

        def agregar_bloque(filas, asiento_izq, asiento_der):

            # Primera pasada: la primera fila del grupo entra
            # por la izquierda, la siguiente por la derecha,
            # y así se va alternando.

            lado = asiento_izq

            for fila in filas:

                for p in pasajeros:

                    if (
                        p["fila"] == fila
                        and p["asiento"] == lado
                    ):

                        orden.append(p)

                lado = (
                    asiento_der
                    if lado == asiento_izq
                    else asiento_izq
                )

            # Segunda pasada: se arranca por el lado contrario,
            # así aparece el otro asiento de cada fila, pero
            # nunca inmediatamente después del primero.

            lado = asiento_der

            for fila in filas:

                for p in pasajeros:

                    if (
                        p["fila"] == fila
                        and p["asiento"] == lado
                    ):

                        orden.append(p)

                lado = (
                    asiento_izq
                    if lado == asiento_der
                    else asiento_der
                )

        filas_impares = list(range(25, 0, -2))   # 25, 23, ..., 1
        filas_pares = list(range(24, 0, -2))     # 24, 22, ..., 2

        # VENTANAS (-2 / 2)

        agregar_bloque(filas_impares, -2, 2)
        agregar_bloque(filas_pares, -2, 2)

        # PASILLOS (-1 / 1)

        agregar_bloque(filas_impares, -1, 1)
        agregar_bloque(filas_pares, -1, 1)

        pasajeros = orden


    else:

        raise ValueError(
            "Política no reconocida. "
            "Usar: Random, Back-to-Front, WILMA o Steffen."
        )


    return pasajeros

> Representación de asientos ocupados.

In [ ]:
# ASIENTOS OCUPADOS

def crear_asientos_ocupados():

    asientos_ocupados = {}

    for fila in range(1, 26):

        asientos_ocupados[fila] = {
            -2: None,
            -1: None,
            1: None,
            2: None
        }

    return asientos_ocupados

> Representación de interferencias.

In [ ]:
# INTERFERENCIAS

def pasajero_que_interfiere(
    pasajero,
    asientos_ocupados
):

    fila = pasajero["fila"]
    asiento = pasajero["asiento"]

    # Ventana izquierda:
    # puede bloquearlo el pasajero de -1
    if asiento == -2:

        return asientos_ocupados[fila][-1]


    # Ventana derecha:
    # puede bloquearlo el pasajero de 1
    if asiento == 2:

        return asientos_ocupados[fila][1]


    # Pasillo: nadie lo bloquea
    return None

> Función de simulación de interferencia.

In [8]:
def empezar_interferencia(
    pasajero,
    bloqueador
):

    # Pasajero que quiere entrar
    pasajero["estado"] = "interferencia"
    pasajero["tiempo_estado"] = 0
    pasajero["interfiere_con"] = bloqueador


    # Pasajero sentado que debe levantarse
    bloqueador["estado"] = "interferencia"
    bloqueador["bloqueando_a"] = pasajero
    bloqueador["tiempo_estado"] = 0

> Para representar el modelo visualmente usamos un HTML. 

> La función de debajo guarda los frames para representar donde esta cada persona en cada segundo de la simulación.

In [ ]:
# GUARDAR UN FRAME PARA EL HTML

def guardar_estado(
    pasajeros,
    tiempo
):

    frame = {
        "segundo": tiempo,
        "pasajeros": []
    }


    for p in pasajeros:

        frame["pasajeros"].append({

            "nombre": p["nombre"],

            "fila": p["fila"],

            "asiento": p["asiento"],

            "carry_on": p["carry_on"],

            "posicion": p["posicion"],

            "estado": p["estado"]

        })


    return frame

**Mecánica de la simulación**

La simulación comienza con los 100 pasajeros fuera del avión y avanza segundo a segundo hasta que todos alcanzan el estado `sentado`.

En cada segundo se realizan las siguientes etapas:

1. Se identifican las posiciones actualmente ocupadas en el pasillo.
2. Se actualiza cuánto tiempo lleva libre cada posición.
3. Si la entrada estuvo libre durante los 3 segundos anteriores, ingresa el siguiente pasajero de la fila de embarque.
4. Se actualiza el estado de cada pasajero según corresponda: caminar, guardar equipaje, resolver una interferencia o sentarse.
5. Se guarda el estado completo del sistema para poder reconstruir posteriormente la evolución del embarque.

La variable de interés principal es el **tiempo total transcurrido hasta que los 100 pasajeros se encuentran sentados**.

> Acá está el motor de la simulación: mueve a los pasajeros segundo a segundo hasta que todos terminan sentados.

In [ ]:
# SIMULACIÓN

def simular(
    politica, p=0.5
):

    # CREAR PASAJEROS

    pasajeros = crear_pasajeros(p)

    # ORDENAR SEGÚN POLÍTICA

    pasajeros = ordenar_pasajeros(
        pasajeros,
        politica
    )

    # CREAR ASIENTOS

    asientos_ocupados = (
        crear_asientos_ocupados()
    )

    # tiempo_libre[fila] =
    # segundos consecutivos que la posición
    # del pasillo lleva libre

    tiempo_libre = [0] * 26

    tiempo = 0

    # Próximo pasajero de la fila de embarque

    siguiente_pasajero = 0

    # Historial para la visualización (HTML)

    historial = []

    # Guardamos segundo 0

    historial.append(

        guardar_estado(
            pasajeros,
            tiempo
        )

    )

    # LOOP PRINCIPAL

    while not all(

        pasajero["estado"] == "sentado"

        for pasajero in pasajeros

    ):

        tiempo += 1

        # 1. POSICIONES OCUPADAS

        ocupadas = []

        for pasajero in pasajeros:

            if (

                pasajero["posicion"] is not None

                and pasajero["posicion"] > 0

                and pasajero["estado"] != "sentado"

            ):

                ocupadas.append(
                    pasajero["posicion"]
                )

        # 2. ACTUALIZAR TIEMPO LIBRE

        for fila in range(1, 26):

            if fila in ocupadas:

                tiempo_libre[fila] = 0

            else:

                tiempo_libre[fila] += 1

        # 3. HACER ENTRAR AL PRÓXIMO PASAJERO

        if siguiente_pasajero < len(pasajeros):

            pasajero_entrada = pasajeros[
                siguiente_pasajero
            ]


            if tiempo_libre[1] >= 3:

                pasajero_entrada["posicion"] = 1

                pasajero_entrada["estado"] = (
                    "caminando"
                )

                pasajero_entrada["tiempo_estado"] = 0


                tiempo_libre[1] = 0


                siguiente_pasajero += 1

        # 4. ACTUALIZAR PASAJEROS

        for pasajero in pasajeros:

            # ESPERANDO

            if pasajero["estado"] == "esperando":

                continue

            # CAMINANDO

            if pasajero["estado"] == "caminando":


                pasajero["tiempo_estado"] += 1


                if pasajero["carry_on"]:

                    tiempo_por_fila = 6

                else:

                    tiempo_por_fila = 3

                # YA ESTÁ EN SU FILA

                if (
                    pasajero["posicion"]
                    == pasajero["fila"]
                ):


                    if pasajero["carry_on"]:

                        pasajero["estado"] = (
                            "guardando"
                        )


                    else:

                        bloqueador = (
                            pasajero_que_interfiere(
                                pasajero,
                                asientos_ocupados
                            )
                        )


                        if bloqueador is not None:

                            empezar_interferencia(
                                pasajero,
                                bloqueador
                            )


                        else:

                            pasajero["estado"] = (
                                "sentandose"
                            )


                    pasajero["tiempo_estado"] = 0


                    continue

                # INTENTAR AVANZAR

                siguiente = (
                    pasajero["posicion"]
                    + 1
                )


                if (

                    pasajero["tiempo_estado"]
                    >= tiempo_por_fila

                    and

                    tiempo_libre[siguiente]
                    >= 3

                ):


                    pasajero["posicion"] += 1


                    pasajero["tiempo_estado"] = 0


                    tiempo_libre[
                        pasajero["posicion"]
                    ] = 0

                    # ACABA DE LLEGAR A SU FILA

                    if (

                        pasajero["posicion"]
                        == pasajero["fila"]

                    ):


                        if pasajero["carry_on"]:

                            pasajero["estado"] = (
                                "guardando"
                            )


                        else:

                            bloqueador = (
                                pasajero_que_interfiere(
                                    pasajero,
                                    asientos_ocupados
                                )
                            )


                            if bloqueador is not None:

                                empezar_interferencia(
                                    pasajero,
                                    bloqueador
                                )


                            else:

                                pasajero["estado"] = (
                                    "sentandose"
                                )


                        pasajero["tiempo_estado"] = 0


                continue

            # GUARDANDO CARRY-ON

            if pasajero["estado"] == "guardando":


                pasajero["tiempo_estado"] += 1


                if (

                    pasajero["tiempo_estado"]
                    >= TIEMPO_GUARDAR

                ):


                    bloqueador = (
                        pasajero_que_interfiere(
                            pasajero,
                            asientos_ocupados
                        )
                    )


                    if bloqueador is not None:

                        empezar_interferencia(
                            pasajero,
                            bloqueador
                        )


                    else:

                        pasajero["estado"] = (
                            "sentandose"
                        )


                    pasajero["tiempo_estado"] = 0


                continue

            # INTERFERENCIA

            if pasajero["estado"] == "interferencia":

                # CASO 1:
                # Este es el pasajero que quiere pasar.
                #
                # Ejemplo:
                # B quiere ir a ventana (-2)
                # y A estaba sentado en pasillo (-1)

                if pasajero["interfiere_con"] is not None:

                    pasajero["tiempo_estado"] += 1


                    # Cuando terminan los 8 segundos
                    # de interferencia
                    if (
                        pasajero["tiempo_estado"]
                        >= TIEMPO_INTERFERENCIA
                    ):

                        bloqueador = pasajero[
                            "interfiere_con"
                        ]

                        # B QUEDA SENTADO DIRECTAMENTE

                        pasajero["estado"] = "sentado"

                        pasajero["posicion"] = None

                        pasajero["tiempo_estado"] = 0


                        # Marcamos que B ocupa su asiento

                        asientos_ocupados[
                            pasajero["fila"]
                        ][
                            pasajero["asiento"]
                        ] = pasajero

                        # A TIENE QUE VOLVER A SENTARSE

                        bloqueador["estado"] = "sentandose"

                        bloqueador["tiempo_estado"] = 0

                        bloqueador["bloqueando_a"] = None


                        # Terminamos la relación
                        # de interferencia

                        pasajero["interfiere_con"] = None


                    continue

                # CASO 2:
                # Este es el pasajero que ya estaba sentado
                # y se levantó para dejar pasar.
                #
                # Mientras dura la interferencia no hace nada,
                # porque B controla los 8 segundos.

                if pasajero["bloqueando_a"] is not None:

                    continue

            # SENTÁNDOSE

            if pasajero["estado"] == "sentandose":


                pasajero["tiempo_estado"] += 1


                if (

                    pasajero["tiempo_estado"]
                    >= TIEMPO_SENTARSE

                ):


                    pasajero["estado"] = "sentado"


                    asientos_ocupados[
                        pasajero["fila"]
                    ][
                        pasajero["asiento"]
                    ] = pasajero


                    pasajero["posicion"] = None


                    pasajero["tiempo_estado"] = 0


                continue

        # 5. GUARDAR FRAME

        historial.append(

            guardar_estado(
                pasajeros,
                tiempo
            )

        )


    return tiempo, historial

> Con el codigo de la simulación armado; corremos todos las politicas para ver sus resultados.

In [11]:
for politica in [
    "Random",
    "Back-to-Front",
    "WILMA",
    "Steffen"
]:

    tiempo, historial = simular(
        p=0.5,
        politica=politica
    )

    print(
        politica,
        ":",
        tiempo,
        "segundos",
        "->",
        round(tiempo / 60, 2),
        "minutos"
    )

Random : 1101 segundos -> 18.35 minutos
Back-to-Front : 1208 segundos -> 20.13 minutos
WILMA : 972 segundos -> 16.2 minutos
Steffen : 934 segundos -> 15.57 minutos


> Vemos en los datos que el que menor tiempo de llenado dio fue con la politica Steffen. 

> En el siguiente codigo esta oculta la simulación del HTML.
>> Nos sirve para exportar un video de como se llena cada avión con las distintas politicas.

In [ ]:
# HTML AUTOCONTENIDO

def crear_html(historial, politica, archivo_salida):

    historial_json = json.dumps(
        historial,
        ensure_ascii=False
    )

    html = f"""
<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Boarding Simulation</title>

<style>

:root {{
    --bg: #07090d;
    --panel: #11151d;
    --seat: #171c26;
    --seat-border: #4d596a;

    --walking: #2f80ed;
    --stowing: #f2c94c;
    --interference: #eb5757;
    --sitting: #9b51e0;
    --seated: #27ae60;

    --text: #f5f7fb;
    --muted: #9aa4b2;
}}

* {{
    box-sizing: border-box;
}}

body {{
    margin: 0;
    background: var(--bg);
    color: var(--text);
    font-family: Arial, sans-serif;
}}

.app {{
    height: 100vh;
    display: grid;
    grid-template-columns: 1fr 260px;
    gap: 18px;
    padding: 18px;
}}

.stage {{
    position: relative;
    border: 1px solid #202734;
    border-radius: 18px;
    overflow: hidden;
    background: #090c12;
}}

.header {{
    position: absolute;
    top: 18px;
    left: 22px;
    z-index: 30;
}}

.header h1 {{
    margin: 0;
    font-size: 23px;
    letter-spacing: .08em;
}}

.header p {{
    margin: 6px 0 0;
    color: var(--muted);
    font-size: 13px;
}}

.policy {{
    display: inline-block;
    margin-top: 8px;
    padding: 5px 9px;
    border-radius: 999px;
    border: 1px solid #2d394c;
    background: #182131;
    font-size: 11px;
}}

.plane {{
    position: absolute;

    left: 52%;
    top: 54%;

    transform: translate(-50%, -50%);

    width: 420px;
    height: 88vh;

    max-height: 850px;
    min-height: 620px;

    border: 2px solid #697386;

    border-radius:
        48% 48% 18% 18%
        / 7% 7% 3% 3%;

    background: #0b0f16;
}}

.nose {{
    position: absolute;

    top: -28px;
    left: 50%;

    transform: translateX(-50%);

    width: 82px;
    height: 45px;

    border: 2px solid #697386;
    border-bottom: none;

    border-radius: 50% 50% 0 0;

    background: #0b0f16;
}}

.cabin {{
    position: absolute;

    inset:
        48px
        26px
        34px;

    display: grid;

    grid-template-rows:
        repeat(25, 1fr);

    gap: 2px;
}}

.row {{
    position: relative;

    display: grid;

    grid-template-columns:
        1fr
        1fr
        24px
        1fr
        1fr;

    align-items: center;

    gap: 5px;
}}

.row-number {{
    position: absolute;

    left: -22px;

    width: 16px;

    text-align: right;

    font-size: 8px;

    color: #758092;
}}

.seat {{
    height: 100%;
    max-height: 20px;
    min-height: 12px;

    border-radius: 4px;

    border:
        1px solid
        var(--seat-border);

    background:
        var(--seat);
}}

.aisle {{
    height: 100%;

    border-left:
        1px dashed #283141;

    border-right:
        1px dashed #283141;
}}

.person {{
    position: absolute;

    width: 12px;
    height: 12px;

    margin-left: -6px;
    margin-top: -6px;

    border-radius: 50%;

    z-index: 20;

    transition:
        left 90ms linear,
        top 90ms linear,
        background-color 90ms linear,
        transform 90ms linear;

    box-shadow:
        0 0 0 2px
        rgba(255,255,255,.12);
}}

.person.caminando {{
    background:
        var(--walking);
}}

.person.guardando {{
    background:
        var(--stowing);

    transform:
        scale(1.15);
}}

.person.interferencia {{
    background:
        var(--interference);

    transform:
        scale(1.2);
}}

.person.sentandose {{
    background:
        var(--sitting);
}}

.person.sentado {{
    background:
        var(--seated);
}}

.bag {{
    position: absolute;

    width: 6px;
    height: 6px;

    right: -5px;
    bottom: -3px;

    border-radius: 1px;

    background: white;

    border: 1px solid #10131a;
}}

.queue {{
    position: absolute;

    left: 32px;
    bottom: 35px;

    width: 110px;
    height: 320px;
}}

.queue-title {{
    font-size: 10px;

    color: var(--muted);

    letter-spacing: .1em;
}}

.queue-person {{
    position: absolute;

    left: 45px;

    width: 10px;
    height: 10px;

    border-radius: 50%;

    background: #596273;
}}

.panel {{
    border-radius: 18px;

    border: 1px solid #202734;

    background: var(--panel);

    padding: 18px;

    display: flex;

    flex-direction: column;

    gap: 14px;
}}

.metric {{
    background: #0d1118;

    border: 1px solid #202734;

    border-radius: 12px;

    padding: 14px;
}}

.metric-label {{
    color: var(--muted);

    font-size: 10px;

    letter-spacing: .12em;

    text-transform: uppercase;
}}

.metric-value {{
    margin-top: 5px;

    font-size: 25px;

    font-weight: bold;
}}

.progress {{
    margin-top: 10px;

    height: 8px;

    background: #1b2230;

    border-radius: 999px;

    overflow: hidden;
}}

.progress-fill {{
    height: 100%;

    width: 0%;

    background:
        linear-gradient(
            90deg,
            #2f80ed,
            #27ae60
        );
}}

.legend {{
    display: grid;

    gap: 8px;

    margin-top: 9px;

    font-size: 12px;
}}

.legend-line {{
    display: flex;

    align-items: center;

    gap: 8px;
}}

.legend-dot {{
    width: 10px;
    height: 10px;

    border-radius: 50%;
}}

.controls {{
    margin-top: auto;

    display: grid;

    gap: 9px;
}}

button,
select {{

    width: 100%;

    border: 0;

    border-radius: 9px;

    padding: 11px;

    color: white;

    background: #1d2634;

    cursor: pointer;
}}

button.primary {{
    background: #2563eb;
}}

.big-clock {{
    position: absolute;

    right: 28px;
    top: 22px;

    z-index: 50;

    font-size: 36px;

    font-weight: bold;

    font-variant-numeric:
        tabular-nums;

    color: white;

    background:
        rgba(0,0,0,.40);

    border:
        1px solid
        #273244;

    padding:
        8px 14px;

    border-radius:
        12px;
}}

</style>

</head>

<body>

<div class="app">

<div class="stage">

    <div class="header">

        <h1>
        BOARDING SIMULATION
        </h1>

        <p>
        100 pasajeros ·
        25 filas ·
        4 asientos
        </p>

        <div class="policy">
        {politica}
        </div>

    </div>


    <div
        class="big-clock"
        id="bigClock"
    >
        0 s
    </div>


    <div
        class="queue"
        id="queue"
    >

        <div class="queue-title">
        PRÓXIMOS
        </div>

    </div>


    <div
        class="plane"
        id="plane"
    >

        <div class="nose">
        </div>

        <div
            class="cabin"
            id="cabin"
        >
        </div>

    </div>

</div>


<div class="panel">

    <div class="metric">

        <div class="metric-label">
        Segundo simulado
        </div>

        <div
            class="metric-value"
            id="timeValue"
        >
        0 s
        </div>

    </div>


    <div class="metric">

        <div class="metric-label">
        Sentados
        </div>

        <div class="metric-value">

            <span id="seatedValue">
            0
            </span>

            / 100

        </div>

        <div class="progress">

            <div
                class="progress-fill"
                id="progressBar"
            >
            </div>

        </div>

    </div>


    <div class="metric">

        <div class="metric-label">
        En el pasillo
        </div>

        <div
            class="metric-value"
            id="aisleValue"
        >
        0
        </div>

    </div>


    <div class="metric">

        <div class="metric-label">
        Esperando
        </div>

        <div
            class="metric-value"
            id="waitingValue"
        >
        100
        </div>

    </div>


    <div class="metric">

        <div class="metric-label">
        Estados
        </div>

        <div class="legend">

            <div class="legend-line">
                <span
                    class="legend-dot"
                    style="background:#2f80ed"
                ></span>
                Caminando
            </div>

            <div class="legend-line">
                <span
                    class="legend-dot"
                    style="background:#f2c94c"
                ></span>
                Guardando carry-on
            </div>

            <div class="legend-line">
                <span
                    class="legend-dot"
                    style="background:#eb5757"
                ></span>
                Interferencia
            </div>

            <div class="legend-line">
                <span
                    class="legend-dot"
                    style="background:#9b51e0"
                ></span>
                Sentándose
            </div>

            <div class="legend-line">
                <span
                    class="legend-dot"
                    style="background:#27ae60"
                ></span>
                Sentado
            </div>

        </div>

    </div>


    <div class="controls">

        <select id="speedSelect">

            <option value="250">
            Lento
            </option>

            <option
                value="100"
                selected
            >
            Normal
            </option>

            <option value="45">
            Rápido
            </option>

        </select>

        <button
            class="primary"
            id="playBtn"
        >
        ▶ Reproducir
        </button>

        <button id="resetBtn">
        ↻ Reiniciar
        </button>

    </div>

</div>

</div>


<script>

/*
El historial real viene de Python.
No hay una segunda simulación en JavaScript.
*/

const historial =
{historial_json};


const cabin =
document.getElementById(
    "cabin"
);

const plane =
document.getElementById(
    "plane"
);

const queue =
document.getElementById(
    "queue"
);

const timeValue =
document.getElementById(
    "timeValue"
);

const bigClock =
document.getElementById(
    "bigClock"
);

const seatedValue =
document.getElementById(
    "seatedValue"
);

const aisleValue =
document.getElementById(
    "aisleValue"
);

const waitingValue =
document.getElementById(
    "waitingValue"
);

const progressBar =
document.getElementById(
    "progressBar"
);

const playBtn =
document.getElementById(
    "playBtn"
);

const resetBtn =
document.getElementById(
    "resetBtn"
);

const speedSelect =
document.getElementById(
    "speedSelect"
);


const seatOrder =
[-2, -1, 1, 2];


let frameActual = 0;

let reproduciendo = false;

let timer = null;


for (
    let fila = 1;
    fila <= 25;
    fila++
) {{

    const row =
    document.createElement(
        "div"
    );

    row.className =
    "row";

    row.dataset.row =
    fila;


    const numero =
    document.createElement(
        "span"
    );

    numero.className =
    "row-number";

    numero.textContent =
    fila;

    row.appendChild(
        numero
    );


    [-2, -1].forEach(
        asiento => {{

            const seat =
            document.createElement(
                "div"
            );

            seat.className =
            "seat";

            seat.dataset.row =
            fila;

            seat.dataset.seat =
            asiento;

            row.appendChild(
                seat
            );
        }}
    );


    const aisle =
    document.createElement(
        "div"
    );

    aisle.className =
    "aisle";

    row.appendChild(
        aisle
    );


    [1, 2].forEach(
        asiento => {{

            const seat =
            document.createElement(
                "div"
            );

            seat.className =
            "seat";

            seat.dataset.row =
            fila;

            seat.dataset.seat =
            asiento;

            row.appendChild(
                seat
            );
        }}
    );


    cabin.appendChild(
        row
    );
}}


const nombres =
historial[0]
.pasajeros
.map(
    p => p.nombre
);


for (
    const nombre
    of nombres
) {{

    const persona =
    document.createElement(
        "div"
    );

    persona.className =
    "person";

    persona.id =
    `persona-${{nombre}}`;

    persona.style.display =
    "none";

    plane.appendChild(
        persona
    );
}}


function posicionPasillo(
    fila
) {{

    const row =
    cabin.querySelector(
        `.row[data-row="${{fila}}"]`
    );

    const rowRect =
    row.getBoundingClientRect();

    const planeRect =
    plane.getBoundingClientRect();

    return {{

        left:
        plane.clientWidth / 2,

        top:
        rowRect.top
        - planeRect.top
        + rowRect.height / 2

    }};
}}


function posicionAsiento(
    fila,
    asiento
) {{

    const seat =
    cabin.querySelector(
        `.seat[data-row="${{fila}}"][data-seat="${{asiento}}"]`
    );

    const rect =
    seat.getBoundingClientRect();

    const planeRect =
    plane.getBoundingClientRect();

    return {{

        left:
        rect.left
        - planeRect.left
        + rect.width / 2,

        top:
        rect.top
        - planeRect.top
        + rect.height / 2

    }};
}}


function actualizarCola(
    pasajeros
) {{

    queue
    .querySelectorAll(
        ".queue-person"
    )
    .forEach(
        el => el.remove()
    );


    pasajeros
    .filter(
        p =>
        p.estado ===
        "esperando"
    )
    .slice(
        0,
        14
    )
    .forEach(
        (p, indice) => {{

            const el =
            document.createElement(
                "div"
            );

            el.className =
            "queue-person";

            el.style.top =
            `${{28 + indice * 19}}px`;


            if (
                p.carry_on
            ) {{

                const bag =
                document.createElement(
                    "div"
                );

                bag.className =
                "bag";

                el.appendChild(
                    bag
                );
            }}


            queue.appendChild(
                el
            );
        }}
    );
}}


function actualizarPasajero(
    p
) {{

    const el =
    document.getElementById(
        `persona-${{p.nombre}}`
    );


    el.className =
    `person ${{p.estado}}`;


    if (
        p.estado ===
        "esperando"
    ) {{

        el.style.display =
        "none";

        return;
    }}


    el.style.display =
    "block";


    /*
    Creamos o quitamos la
    mini valija.
    */

    let bag =
    el.querySelector(
        ".bag"
    );


    if (
        p.carry_on
        && !bag
    ) {{

        bag =
        document.createElement(
            "div"
        );

        bag.className =
        "bag";

        el.appendChild(
            bag
        );
    }}


    if (
        !p.carry_on
        && bag
    ) {{

        bag.remove();
    }}


    let pos;


    if (
        p.estado ===
        "sentado"
    ) {{

        pos =
        posicionAsiento(
            p.fila,
            p.asiento
        );

    }}

    else if (
        p.estado ===
        "sentandose"
        && p.posicion === null
    ) {{

        /*
        Este pasajero ya estaba sentado,
        se levantó por una interferencia
        y ahora está volviendo a sentarse.
        */

        pos =
        posicionAsiento(
            p.fila,
            p.asiento
        );

    }}

    else if (
        p.estado ===
        "interferencia"
        && p.posicion === null
    ) {{

        /*
        Este pasajero ya estaba sentado
        y tuvo que levantarse para dejar pasar.
        */

        pos =
        posicionAsiento(
            p.fila,
            p.asiento
        );

    }}

    else if (
        p.posicion !== null
        && p.posicion > 0
    ) {{

        pos =
        posicionPasillo(
            p.posicion
        );

    }}

    else {{

        return;
    }}


    el.style.left =
    `${{pos.left}}px`;

    el.style.top =
    `${{pos.top}}px`;
}}


function mostrarFrame(
    indice
) {{

    const frame =
    historial[indice];


    const pasajeros =
    frame.pasajeros;


    /*
    El segundo mostrado es
    exactamente el segundo
    guardado por Python.
    */

    timeValue.textContent =
    `${{frame.segundo}} s`;

    bigClock.textContent =
    `${{frame.segundo}} s`;


    let sentados = 0;

    let esperando = 0;

    let pasillo = 0;


    for (
        const p
        of pasajeros
    ) {{

        actualizarPasajero(
            p
        );


        if (
            p.estado ===
            "sentado"
        ) {{
            sentados++;
        }}


        if (
            p.estado ===
            "esperando"
        ) {{
            esperando++;
        }}


        if (
            [
                "caminando",
                "guardando",
                "interferencia",
                "sentandose"
            ].includes(
                p.estado
            )
            &&
            p.posicion !== null
        ) {{
            pasillo++;
        }}
    }}


    seatedValue.textContent =
    sentados;

    waitingValue.textContent =
    esperando;

    aisleValue.textContent =
    pasillo;

    progressBar.style.width =
    `${{sentados}}%`;


    actualizarCola(
        pasajeros
    );
}}


function avanzar() {{

    if (
        frameActual
        >= historial.length
    ) {{

        pausar();

        return;
    }}


    mostrarFrame(
        frameActual
    );


    frameActual++;
}}


function reproducir() {{

    if (
        reproduciendo
    ) {{
        return;
    }}


    reproduciendo =
    true;


    playBtn.textContent =
    "⏸ Pausar";


    const velocidad =
    Number(
        speedSelect.value
    );


    timer =
    setInterval(
        avanzar,
        velocidad
    );
}}


function pausar() {{

    reproduciendo =
    false;


    clearInterval(
        timer
    );


    timer =
    null;


    playBtn.textContent =
    "▶ Reproducir";
}}


function reiniciar() {{

    pausar();


    frameActual =
    0;


    mostrarFrame(
        0
    );
}}


playBtn
.addEventListener(
    "click",
    () => {{

        if (
            reproduciendo
        ) {{
            pausar();
        }}

        else {{
            reproducir();
        }}
    }}
);


resetBtn
.addEventListener(
    "click",
    reiniciar
);


speedSelect
.addEventListener(
    "change",
    () => {{

        if (
            reproduciendo
        ) {{

            pausar();

            reproducir();
        }}
    }}
);


window
.addEventListener(
    "resize",
    () => {{

        mostrarFrame(
            Math.min(
                frameActual,
                historial.length - 1
            )
        );
    }}
);


mostrarFrame(
    0
);

</script>

</body>
</html>
"""

    Path(
        archivo_salida
    ).write_text(
        html,
        encoding="utf-8"
    )

> Corremos las 4 políticas una vez cada una y generamos su animación en HTML.

In [ ]:
# EJECUTAR LAS 4 POLÍTICAS

politicas = [
    "Random",
    "Back-to-Front",
    "WILMA",
    "Steffen"
]


for politica in politicas:

    # SIMULAR ESTA POLÍTICA

    tiempo_total, historial = simular(
        p=P_CARRY_ON,
        politica=politica
    )


    # MOSTRAR RESULTADO

    print("\n-----------------------------")
    print("Política:", politica)
    print("-----------------------------")

    print(
        "Tiempo total:",
        tiempo_total,
        "segundos"
    )

    print(
        "Tiempo total:",
        round(
            tiempo_total / 60,
            2
        ),
        "minutos"
    )

    # CREAR NOMBRE DEL HTML

    nombre_archivo = (
        politica
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    archivo_html = (
        f"boarding_{nombre_archivo}_animado.html"
    )

    # CREAR HTML

    crear_html(
        historial,
        politica,
        archivo_html
    )


    print(
        "HTML creado:",
        archivo_html
    )


print(
    "\nSe generaron las 4 animaciones."
)


-----------------------------
Política: Random
-----------------------------
Tiempo total: 1127 segundos
Tiempo total: 18.78 minutos
HTML creado: boarding_random_animado.html

-----------------------------
Política: Back-to-Front
-----------------------------
Tiempo total: 1186 segundos
Tiempo total: 19.77 minutos
HTML creado: boarding_back_to_front_animado.html

-----------------------------
Política: WILMA
-----------------------------
Tiempo total: 1035 segundos
Tiempo total: 17.25 minutos
HTML creado: boarding_wilma_animado.html

-----------------------------
Política: Steffen
-----------------------------
Tiempo total: 945 segundos
Tiempo total: 15.75 minutos
HTML creado: boarding_steffen_animado.html

Se generaron las 4 animaciones.


> Las cuatro simulaciones estaran adjuntadas en la carpeta de entrega del trabajo practico.

**Simulación de Monte Carlo**

Una única corrida no es suficiente para comparar las políticas de embarque, ya que el resultado depende de componentes aleatorios, como la asignación de carry-on y el orden de pasajeros dentro de determinadas políticas.

Por este motivo, repetimos la simulación $N$ veces para cada política.

Si $T_i$ representa el tiempo total obtenido en la simulación $i$, estimamos el tiempo esperado mediante:

$$
\hat{E}[T] = \bar{T} = \frac{1}{N}\sum_{i=1}^{N} T_i
$$

También calculamos la **desviación estándar muestral**:

$$
s = \sqrt{\frac{1}{N-1}\sum_{i=1}^{N}(T_i-\bar{T})^2}
$$

y el **error estándar de la media**:

$$
SE(\bar{T}) = \frac{s}{\sqrt{N}}
$$

El error estándar permite cuantificar la precisión de nuestra estimación del tiempo esperado. A medida que aumenta el número de simulaciones, este error disminuye aproximadamente a una tasa proporcional a:

$$
\frac{1}{\sqrt{N}}
$$

> Corremos cada política 50 veces (para distintas probabilidades de carry-on) y calculamos el promedio, el desvío y el intervalo de confianza.

In [ ]:
import numpy as np
import pandas as pd

# PARÁMETROS DE MONTE CARLO

politicas = [
    "Random",
    "Back-to-Front",
    "WILMA",
    "Steffen"
]

# Numero de simulaciones de Monte Carlo que vamos a realizar.
N_SIMULACIONES = 50

# Probabilidades de carry-on a evaluar (p=0.5, p=1 y p=0)
valores_p = [0.5, 1.0, 0.0]

# DICCIONARIO PARA GUARDAR RESULTADOS (por p y por política)

resultados_mc = {}

# MONTE CARLO

for p in valores_p:

    resultados_mc[p] = {}

    for politica in politicas:

        tiempos = []

        print(f"\nSimulando (p={p}):", politica)

        for i in range(N_SIMULACIONES):

            # simular devuelve:
            # tiempo total, historial
            
            # El historial no lo necesitamos para Monte Carlo,
            # por eso usamos "_"

            tiempo, _ = simular(
                p=p,
                politica=politica
            )

            tiempos.append(tiempo)

        # ESTADÍSTICAS

        media = np.mean(tiempos)

        desvio = np.std(
            tiempos,
            ddof=1
        )

        error_estandar = (
            desvio
            / np.sqrt(N_SIMULACIONES)
        )

        # Intervalo de confianza aproximado del 95%

        ic_inferior = (
            media
            - 1.96 * error_estandar
        )

        ic_superior = (
            media
            + 1.96 * error_estandar
        )

        # GUARDAR RESULTADOS

        resultados_mc[p][politica] = {
            "media": media,
            "desvio": desvio,
            "error_estandar": error_estandar,
            "ic_inferior": ic_inferior,
            "ic_superior": ic_superior,
            "tiempos": tiempos
        }

# MOSTRAR RESULTADOS

for p in valores_p:

    for politica in politicas:

        r = resultados_mc[p][politica]

        print("\n--------------------------------")
        print(f"Política: {politica}  (p={p})")
        print("--------------------------------")

        print(
            "Tiempo promedio:",
            round(r["media"], 2),
            "segundos"
        )

        print(
            "Tiempo promedio:",
            round(r["media"] / 60, 2),
            "minutos"
        )

        print(
            "Desvío estándar:",
            round(r["desvio"], 2),
            "segundos"
        )

        print(
            "Error estándar:",
            round(r["error_estandar"], 2),
            "segundos"
        )

        print(
            "IC 95%:",
            round(r["ic_inferior"], 2),
            "-",
            round(r["ic_superior"], 2),
            "segundos"
        )


# CREAR TABLA RESUMEN

tabla = []

for p in valores_p:

    for politica in politicas:

        r = resultados_mc[p][politica]

        tabla.append({
            "p (carry-on)": p,
            "Política": politica,
            "Media (s)": r["media"],
            "Media (min)": r["media"] / 60,
            "Desvío estándar (s)": r["desvio"],
            "Error estándar (s)": r["error_estandar"],
            "IC 95% inferior (s)": r["ic_inferior"],
            "IC 95% superior (s)": r["ic_superior"]
        })


df_resultados = pd.DataFrame(tabla)

# REDONDEAR PARA MOSTRAR

df_resultados_redondeado = df_resultados.copy()

columnas_numericas = [
    "Media (s)",
    "Media (min)",
    "Desvío estándar (s)",
    "Error estándar (s)",
    "IC 95% inferior (s)",
    "IC 95% superior (s)"
]

df_resultados_redondeado[
    columnas_numericas
] = df_resultados_redondeado[
    columnas_numericas
].round(2)

# VER TABLA
df_resultados_redondeado


Simulando (p=0.5): Random

Simulando (p=0.5): Back-to-Front

Simulando (p=0.5): WILMA

Simulando (p=0.5): Steffen

Simulando (p=1.0): Random

Simulando (p=1.0): Back-to-Front

Simulando (p=1.0): WILMA

Simulando (p=1.0): Steffen

Simulando (p=0.0): Random

Simulando (p=0.0): Back-to-Front

Simulando (p=0.0): WILMA

Simulando (p=0.0): Steffen

--------------------------------
Política: Random  (p=0.5)
--------------------------------
Tiempo promedio: 1089.86 segundos
Tiempo promedio: 18.16 minutos
Desvío estándar: 32.68 segundos
Error estándar: 4.62 segundos
IC 95%: 1080.8 - 1098.92 segundos

--------------------------------
Política: Back-to-Front  (p=0.5)
--------------------------------
Tiempo promedio: 1219.96 segundos
Tiempo promedio: 20.33 minutos
Desvío estándar: 39.65 segundos
Error estándar: 5.61 segundos
IC 95%: 1208.97 - 1230.95 segundos

--------------------------------
Política: WILMA  (p=0.5)
--------------------------------
Tiempo promedio: 1069.36 segundos
Tiempo promed

,p (carry-on),Política,Media (s),Media (min),Desvío estándar (s),Error estándar (s),IC 95% inferior (s),IC 95% superior (s)
0,0.5,Random,1089.86,18.16,32.68,4.62,1080.80,1098.92
1,0.5,Back-to-Front,1219.96,20.33,39.65,5.61,1208.97,1230.95
2,0.5,WILMA,1069.36,17.82,40.71,5.76,1058.07,1080.65
3,0.5,Steffen,943.16,15.72,17.47,2.47,938.32,948.00
4,1.0,Random,1233.90,20.56,29.64,4.19,1225.68,1242.12
5,1.0,Back-to-Front,1561.40,26.02,11.60,1.64,1558.18,1564.62
6,1.0,WILMA,1197.70,19.96,26.73,3.78,1190.29,1205.11
7,1.0,Steffen,998.00,16.63,0.00,0.00,998.00,998.00
8,0.0,Random,700.08,11.67,19.35,2.74,694.72,705.44
9,0.0,Back-to-Front,729.16,12.15,11.72,1.66,725.91,732.41


> Grafico comparativo de politicas de embarque vs. tiempo promedio de llenado (utilizando la libreria matplotlib).

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))

ancho_barra = 0.25
x = np.arange(len(politicas))

for i, p in enumerate(valores_p):

    medias = [
        resultados_mc[p][politica]["media"] / 60
        for politica in politicas
    ]

    errores = [
        (
            resultados_mc[p][politica]["ic_superior"]
            - resultados_mc[p][politica]["ic_inferior"]
        ) / 2 / 60
        for politica in politicas
    ]

    ax.bar(
        x + i * ancho_barra,
        medias,
        ancho_barra,
        yerr=errores,
        capsize=4,
        label=f"p = {p}"
    )

ax.set_xlabel("Política de embarque")
ax.set_ylabel("Tiempo promedio (minutos)")
ax.set_title("Comparación de políticas de embarque (Monte Carlo, IC 95%)")
ax.set_xticks(x + ancho_barra)
ax.set_xticklabels(politicas)
ax.legend(title="Prob. carry-on (p)")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

> Grafico comparando tiempos por politica, teniendo en cuenta las probabilidades de tener carry on.

In [ ]:
fig, axes = plt.subplots(1, len(valores_p), figsize=(15, 5), sharey=True)

for ax, p in zip(axes, valores_p):

    datos_minutos = [
        [t / 60 for t in resultados_mc[p][politica]["tiempos"]]
        for politica in politicas
    ]

    ax.boxplot(datos_minutos)
    ax.set_xticklabels(politicas)
    ax.set_title(f"p = {p}")
    ax.set_xlabel("Política")
    ax.tick_params(axis="x", rotation=45)
    ax.grid(axis="y", alpha=0.3)

axes[0].set_ylabel("Tiempo (minutos)")

fig.suptitle("Distribución de tiempos por política y probabilidad de carry-on")
plt.tight_layout()
plt.show()

**BONUS — 30%: clases, grupos de embarque, tipos de pasajero y modalidad de vuelo**

Extensión libre del modelo (no modifica nada de lo anterior):

- **Clases y grupos de embarque (bloques contiguos de filas):** filas 1-5 = "Primera Clase", siempre embarcan primero. El resto se reparte en bloques contiguos según `[0.3, 0.3, 0.4]`: grupo1 = filas 6-11 (tarifa más alta, entra primero), grupo2 = filas 12-17, grupo3 = filas 18-25 (tarifa más baja, entra último) — como en una aerolínea real, donde la fila determina el grupo de embarque.
- **Orden dentro de cada grupo:** configurable con `orden_intragrupo`, `"random"` (aleatorio) o `"steffen"` (ventanas antes que pasillos, alternando filas pares/impares, igual que la política Steffen original pero aplicada solo a las filas de ese bloque).
- **Tipos de pasajero:** cada pasajero es `discapacidad` (3%, prioridad de pre-boarding antes que nadie), `lento` (25%), `rapido` (25%) o `normal` (47%), cada uno con un multiplicador de velocidad que afecta tanto el tiempo de caminata por fila como el tiempo de guardado de carry-on.
- **Modalidad de vuelo:** `"corto"` (p_carry_on=0.75, se evita despachar equipaje) vs `"largo"` (p_carry_on=0.35, se factura más equipaje).

Se compara la nueva política **Premium-Prioridad** (con sus dos variantes de orden intra-grupo) contra `Random` y `Steffen` bajo ambas modalidades.

> Parámetros del bonus: dividimos el avión en primera clase y tres grupos de embarque económico.

In [ ]:
# BONUS: CLASES, GRUPOS DE EMBARQUE, TIPOS DE PASAJERO
# Y MODALIDAD DE VUELO (CORTO / LARGO)

# PARÁMETROS DEL BONUS

# Proporciones de los 3 grupos de embarque económico (filas 6-25),
# repartidas como BLOQUES CONTIGUOS de filas (no al azar por persona):
# las filas justo después de primera clase son grupo1 (tarifa más
# alta), después grupo2, y las del fondo grupo3 (tarifa más baja).
PROPORCIONES_GRUPOS = [0.3, 0.3, 0.4]

FILAS_PRIMERA = list(range(1, 6))          # filas 1-5
FILAS_ECONOMICAS = list(range(6, 26))      # filas 6-25 (20 filas)

_n_filas = len(FILAS_ECONOMICAS)
_n1 = round(_n_filas * PROPORCIONES_GRUPOS[0])
_n2 = round(_n_filas * PROPORCIONES_GRUPOS[1])

FILAS_GRUPO1 = FILAS_ECONOMICAS[:_n1]
FILAS_GRUPO2 = FILAS_ECONOMICAS[_n1:_n1 + _n2]
FILAS_GRUPO3 = FILAS_ECONOMICAS[_n1 + _n2:]

> Función que devuelve la clase o grupo de embarque según la fila del pasajero.

In [16]:
def clase_de_fila(fila):

    if fila in FILAS_PRIMERA:
        return "primera_clase"

    if fila in FILAS_GRUPO1:
        return "grupo1"

    if fila in FILAS_GRUPO2:
        return "grupo2"

    return "grupo3"

> Las personas no somos todas iguales. Asi que, decidimos poner probabilidades a tener distintos tipos de personas en el avión.

>> Gente con discapacidad ó embarazadas (entran primero al avión).
>> Gente que sube rapido ó gente que sube lento (mayores de edad).

>> También, le pusimos probabilidad a tener carry-on en vuelos de largo plazo vs. vuelos de corto plazo.


In [17]:
# Probabilidades de cada tipo de pasajero
P_DISCAPACIDAD = 0.03
P_LENTO = 0.25
P_RAPIDO = 0.25
# el resto (1 - suma anterior) queda como "normal"

# Multiplicador de tiempo (caminata Y guardado de carry-on)
# según el tipo de pasajero
MULT_VELOCIDAD = {
    "discapacidad": 2.0,
    "lento": 1.5,
    "rapido": 0.75,
    "normal": 1.0,
}

# Probabilidad de carry-on según modalidad de vuelo
P_CARRY_ON_MODALIDAD = {
    "corto": 0.75,  # vuelos cortos: se evita el despacho de equipaje
    "largo": 0.35,  # vuelos largos: se factura más, se prioriza viajar liviano en el pasillo
}

> Versión extendida de `crear_pasajeros`: ahora cada pasajero también tiene un tipo (normal, lento, rápido o discapacidad) y una clase según su fila.

In [ ]:
# CREAR PASAJEROS (con clase, grupo, tipo y carry-on por modalidad)

def crear_pasajeros_bonus(modalidad="corto"):

    p_carry_on = P_CARRY_ON_MODALIDAD[modalidad]

    pasajeros = []
    contador = 1

    for fila in range(1, 26):

        for asiento in [-2, -1, 1, 2]:

            #  tipo de pasajero 
            r = random.random()

            if r < P_DISCAPACIDAD:
                tipo = "discapacidad"
            elif r < P_DISCAPACIDAD + P_LENTO:
                tipo = "lento"
            elif r < P_DISCAPACIDAD + P_LENTO + P_RAPIDO:
                tipo = "rapido"
            else:
                tipo = "normal"

            #  clase / grupo de embarque (según la fila) 
            clase = clase_de_fila(fila)

            pasajero = {
                "nombre": f"B{contador}",
                "fila": fila,
                "asiento": asiento,
                "carry_on": random.random() < p_carry_on,
                "posicion": 0,
                "estado": "esperando",
                "tiempo_estado": 0,

                "interfiere_con": None,
                "bloqueando_a": None,

                "tipo": tipo,
                "clase": clase,
                "mult_velocidad": MULT_VELOCIDAD[tipo],
            }

            pasajeros.append(pasajero)
            contador += 1

    return pasajeros

> Función auxiliar que evita que dos pasajeros consecutivos queden en la misma fila dentro de un mismo bloque.

In [ ]:
# ORDEN DENTRO DE UN BLOQUE (clase/grupo)
#
# "random"  -> orden aleatorio, como antes.
# "steffen" -> misma lógica que la política Steffen original
#              (ventanas antes que pasillos, alternando filas
#              impares/pares) pero aplicada solo a las filas
#              de ese bloque.

def _separar_filas_consecutivas(orden):

    # Reparación de seguridad: si dos pasajeros consecutivos
    # quedaron en la misma fila (puede pasar en bloques chicos,
    # cuando el pasajero con discapacidad de esa fila se sacó
    # del bloque y "corrió" el armado de las dos pasadas), se
    # intercambia al segundo de ellos con algún otro pasajero
    # del bloque que sí pueda convivir con los vecinos de ambas
    # posiciones. Al ser un intercambio (swap), el largo de la
    # lista nunca cambia y el barrido de la posición candidata
    # "j" recorre range(n) una sola vez por conflicto: por
    # construcción esto SIEMPRE termina (no hay forma de que
    # quede dando vueltas).

    orden = list(orden)

    n = len(orden)

    for i in range(1, n):

        if orden[i]["fila"] != orden[i - 1]["fila"]:
            continue

        for j in range(n):

            if j in (i - 1, i, i + 1):
                continue

            candidato = orden[j]
            actual = orden[i]

            # candidato pasaría a ocupar la posición i
            if candidato["fila"] == orden[i - 1]["fila"]:
                continue
            if i + 1 < n and candidato["fila"] == orden[i + 1]["fila"]:
                continue

            # actual pasaría a ocupar la posición j
            if j - 1 >= 0 and orden[j - 1]["fila"] == actual["fila"]:
                continue
            if j + 1 < n and orden[j + 1]["fila"] == actual["fila"]:
                continue

            orden[i], orden[j] = orden[j], orden[i]
            break

        # Si no se encontró ningún intercambio válido, el
        # conflicto queda (solo puede pasar si ese bloque tiene
        # una única fila con sus dos asientos aislados de
        # cualquier otra fila, caso en el que es matemáticamente
        # imposible separarlos); en cualquier otro caso, con la
        # proporción real de pasajeros con discapacidad, esto no
        # ocurre.

    return orden

> Ordena a los pasajeros dentro de un mismo bloque (clase o grupo), ya sea al azar o con la lógica de Steffen.

In [20]:
def ordenar_bloque(bloque, modo="random"):

    if modo == "random":
        random.shuffle(bloque)
        return bloque

    if modo == "steffen":

        # Misma lógica que la política Steffen corregida
        # (ventanas antes que pasillos, separando filas
        # impares/pares y alternando el lado en DOS pasadas
        # para que dos pasajeros consecutivos nunca compartan
        # fila), pero aplicada solo sobre las filas de este
        # bloque de embarque.

        filas_bloque = sorted(
            {p["fila"] for p in bloque},
            reverse=True
        )

        filas_impares = [f for f in filas_bloque if f % 2 == 1]
        filas_pares = [f for f in filas_bloque if f % 2 == 0]

        orden = []

        def agregar_grupo_asiento(filas, asiento_izq, asiento_der):

            # Primera pasada: arranca por la izquierda y
            # alterna de lado en cada fila.

            lado = asiento_izq

            for fila in filas:

                for p in bloque:

                    if (
                        p["fila"] == fila
                        and p["asiento"] == lado
                    ):

                        orden.append(p)

                lado = (
                    asiento_der
                    if lado == asiento_izq
                    else asiento_izq
                )

            # Segunda pasada: arranca por el lado contrario,
            # para que aparezca el otro asiento de cada fila
            # sin quedar pegado al de la primera pasada.

            lado = asiento_der

            for fila in filas:

                for p in bloque:

                    if (
                        p["fila"] == fila
                        and p["asiento"] == lado
                    ):

                        orden.append(p)

                lado = (
                    asiento_izq
                    if lado == asiento_der
                    else asiento_der
                )

        # VENTANAS (-2 / 2)
        agregar_grupo_asiento(filas_impares, -2, 2)
        agregar_grupo_asiento(filas_pares, -2, 2)

        # PASILLOS (-1 / 1)
        agregar_grupo_asiento(filas_impares, -1, 1)
        agregar_grupo_asiento(filas_pares, -1, 1)

        # Reparamos los pocos casos borde donde, por los huecos
        # de discapacidad, dos pasajeros consecutivos terminaron
        # en la misma fila (ver celda de EDA / verificación).
        orden = _separar_filas_consecutivas(orden)

        return orden

    raise ValueError("modo debe ser 'random' o 'steffen'")

> Política nueva "Premium-Prioridad": entran primero las personas con discapacidad, después primera clase y después los tres grupos económicos en orden.

In [21]:
# --------------------------------------------------------------
# POLÍTICA DE EMBARQUE "PREMIUM-PRIORIDAD"
#
# Orden: pre-boarding discapacidad -> primera clase (filas 1-5)
# -> grupo1 -> grupo2 -> grupo3 (bloques contiguos de filas,
# grupo1 = filas justo después de primera, grupo3 = las del
# fondo). Dentro de cada bloque, el orden se elige con
# `orden_intragrupo`: "random" o "steffen".
# --------------------------------------------------------------

def ordenar_pasajeros_bonus(pasajeros, orden_intragrupo="random"):

    discapacitados = [p for p in pasajeros if p["tipo"] == "discapacidad"]
    random.shuffle(discapacitados)

    primera = [
        p for p in pasajeros
        if p["clase"] == "primera_clase" and p["tipo"] != "discapacidad"
    ]

    grupo1 = [
        p for p in pasajeros
        if p["clase"] == "grupo1" and p["tipo"] != "discapacidad"
    ]

    grupo2 = [
        p for p in pasajeros
        if p["clase"] == "grupo2" and p["tipo"] != "discapacidad"
    ]

    grupo3 = [
        p for p in pasajeros
        if p["clase"] == "grupo3" and p["tipo"] != "discapacidad"
    ]

    primera = ordenar_bloque(primera, orden_intragrupo)
    grupo1 = ordenar_bloque(grupo1, orden_intragrupo)
    grupo2 = ordenar_bloque(grupo2, orden_intragrupo)
    grupo3 = ordenar_bloque(grupo3, orden_intragrupo)

    return discapacitados + primera + grupo1 + grupo2 + grupo3

> Igual que `guardar_estado`, pero guardando también el tipo y la clase de cada pasajero para poder pintarlos en la animación.

In [22]:
# --------------------------------------------------------------
# GUARDAR UN FRAME PARA EL HTML DEL BONUS
#
# Igual que `guardar_estado`, pero agrega "tipo" y "clase" para
# que la visualización pueda pintar los asientos por clase y
# distinguir a los pasajeros con discapacidad.
# --------------------------------------------------------------

def guardar_estado_bonus(pasajeros, tiempo):

    frame = {
        "segundo": tiempo,
        "pasajeros": []
    }

    for p in pasajeros:

        frame["pasajeros"].append({
            "nombre": p["nombre"],
            "fila": p["fila"],
            "asiento": p["asiento"],
            "carry_on": p["carry_on"],
            "posicion": p["posicion"],
            "estado": p["estado"],
            "tipo": p["tipo"],
            "clase": p["clase"],
        })

    return frame

> Misma lógica que `simular`, pero el tiempo de cada acción se ajusta según qué tan rápido o lento es cada tipo de pasajero.

In [23]:
# --------------------------------------------------------------
# SIMULACIÓN BONUS
#
# Copia de `simular` (celda de mecánica de simulación) donde el
# tiempo por fila y el tiempo de guardado de carry-on se escalan
# por el multiplicador de velocidad del tipo de pasajero.
# --------------------------------------------------------------

def simular_bonus(modalidad="corto", orden_intragrupo="random"):

    pasajeros = crear_pasajeros_bonus(modalidad)
    pasajeros = ordenar_pasajeros_bonus(pasajeros, orden_intragrupo)

    asientos_ocupados = crear_asientos_ocupados()

    tiempo_libre = [0] * 26
    tiempo = 0
    siguiente_pasajero = 0
    historial = []

    historial.append(guardar_estado_bonus(pasajeros, tiempo))

    while not all(p["estado"] == "sentado" for p in pasajeros):

        tiempo += 1

        # 1. posiciones ocupadas
        ocupadas = [
            p["posicion"] for p in pasajeros
            if p["posicion"] is not None and p["posicion"] > 0 and p["estado"] != "sentado"
        ]

        # 2. actualizar tiempo libre
        for fila in range(1, 26):
            if fila in ocupadas:
                tiempo_libre[fila] = 0
            else:
                tiempo_libre[fila] += 1

        # 3. hacer entrar al próximo pasajero
        if siguiente_pasajero < len(pasajeros):
            pasajero_entrada = pasajeros[siguiente_pasajero]

            if tiempo_libre[1] >= 3:
                pasajero_entrada["posicion"] = 1
                pasajero_entrada["estado"] = "caminando"
                pasajero_entrada["tiempo_estado"] = 0
                tiempo_libre[1] = 0
                siguiente_pasajero += 1

        # 4. actualizar pasajeros
        for pasajero in pasajeros:

            if pasajero["estado"] == "esperando":
                continue

            if pasajero["estado"] == "caminando":

                pasajero["tiempo_estado"] += 1

                tiempo_base = 6 if pasajero["carry_on"] else 3
                tiempo_por_fila = tiempo_base * pasajero["mult_velocidad"]

                if pasajero["posicion"] == pasajero["fila"]:

                    if pasajero["carry_on"]:
                        pasajero["estado"] = "guardando"
                    else:
                        bloqueador = pasajero_que_interfiere(pasajero, asientos_ocupados)
                        if bloqueador is not None:
                            empezar_interferencia(pasajero, bloqueador)
                        else:
                            pasajero["estado"] = "sentandose"

                    pasajero["tiempo_estado"] = 0
                    continue

                siguiente = pasajero["posicion"] + 1

                if pasajero["tiempo_estado"] >= tiempo_por_fila and tiempo_libre[siguiente] >= 3:

                    pasajero["posicion"] += 1
                    pasajero["tiempo_estado"] = 0
                    tiempo_libre[pasajero["posicion"]] = 0

                    if pasajero["posicion"] == pasajero["fila"]:

                        if pasajero["carry_on"]:
                            pasajero["estado"] = "guardando"
                        else:
                            bloqueador = pasajero_que_interfiere(pasajero, asientos_ocupados)
                            if bloqueador is not None:
                                empezar_interferencia(pasajero, bloqueador)
                            else:
                                pasajero["estado"] = "sentandose"

                        pasajero["tiempo_estado"] = 0

                continue

            if pasajero["estado"] == "guardando":

                pasajero["tiempo_estado"] += 1

                tiempo_guardar_efectivo = TIEMPO_GUARDAR * pasajero["mult_velocidad"]

                if pasajero["tiempo_estado"] >= tiempo_guardar_efectivo:

                    bloqueador = pasajero_que_interfiere(pasajero, asientos_ocupados)
                    if bloqueador is not None:
                        empezar_interferencia(pasajero, bloqueador)
                    else:
                        pasajero["estado"] = "sentandose"

                    pasajero["tiempo_estado"] = 0

                continue

            if pasajero["estado"] == "interferencia":

                if pasajero["interfiere_con"] is not None:

                    pasajero["tiempo_estado"] += 1

                    if pasajero["tiempo_estado"] >= TIEMPO_INTERFERENCIA:

                        bloqueador = pasajero["interfiere_con"]

                        pasajero["estado"] = "sentado"
                        pasajero["posicion"] = None
                        pasajero["tiempo_estado"] = 0
                        asientos_ocupados[pasajero["fila"]][pasajero["asiento"]] = pasajero

                        bloqueador["estado"] = "sentandose"
                        bloqueador["tiempo_estado"] = 0
                        bloqueador["bloqueando_a"] = None

                        pasajero["interfiere_con"] = None

                    continue

                if pasajero["bloqueando_a"] is not None:
                    continue

            if pasajero["estado"] == "sentandose":

                pasajero["tiempo_estado"] += 1

                if pasajero["tiempo_estado"] >= TIEMPO_SENTARSE:

                    pasajero["estado"] = "sentado"
                    asientos_ocupados[pasajero["fila"]][pasajero["asiento"]] = pasajero
                    pasajero["posicion"] = None
                    pasajero["tiempo_estado"] = 0

                continue

        historial.append(guardar_estado_bonus(pasajeros, tiempo))

    return tiempo, historial

> Comparamos Premium-Prioridad (random y steffen) contra Random y Steffen originales, para vuelo corto y largo.

In [24]:
# --------------------------------------------------------------
# COMPARACIÓN RÁPIDA: Premium-Prioridad (random vs steffen dentro
# de cada grupo) contra las políticas originales, para vuelo
# corto y vuelo largo
# --------------------------------------------------------------

N_SIMULACIONES_BONUS = 300

resultados_bonus = {}

for modalidad in ["corto", "largo"]:

    p_modalidad = P_CARRY_ON_MODALIDAD[modalidad]

    tiempos_premium_random = [
        simular_bonus(modalidad, "random")[0]
        for _ in range(N_SIMULACIONES_BONUS)
    ]

    tiempos_premium_steffen = [
        simular_bonus(modalidad, "steffen")[0]
        for _ in range(N_SIMULACIONES_BONUS)
    ]

    tiempos_random = [simular("Random", p=p_modalidad)[0] for _ in range(N_SIMULACIONES_BONUS)]
    tiempos_steffen = [simular("Steffen", p=p_modalidad)[0] for _ in range(N_SIMULACIONES_BONUS)]

    resultados_bonus[modalidad] = {
        "Premium-Prioridad (intra-grupo random)": (np.mean(tiempos_premium_random), np.std(tiempos_premium_random, ddof=1)),
        "Premium-Prioridad (intra-grupo steffen)": (np.mean(tiempos_premium_steffen), np.std(tiempos_premium_steffen, ddof=1)),
        "Random": (np.mean(tiempos_random), np.std(tiempos_random, ddof=1)),
        "Steffen": (np.mean(tiempos_steffen), np.std(tiempos_steffen, ddof=1)),
    }

    print(f"\nModalidad: {modalidad} (p_carry_on={p_modalidad})")
    for pol, (media, desvio) in resultados_bonus[modalidad].items():
        print(f"  {pol}: media={media:.1f}s ({media/60:.2f} min)  desvio={desvio:.1f}s")


Modalidad: corto (p_carry_on=0.75)
  Premium-Prioridad (intra-grupo random): media=1518.4s (25.31 min)  desvio=41.1s
  Premium-Prioridad (intra-grupo steffen): media=1422.6s (23.71 min)  desvio=38.3s
  Random: media=1173.8s (19.56 min)  desvio=35.3s
  Steffen: media=975.4s (16.26 min)  desvio=12.0s

Modalidad: largo (p_carry_on=0.35)
  Premium-Prioridad (intra-grupo random): media=1318.4s (21.97 min)  desvio=59.0s
  Premium-Prioridad (intra-grupo steffen): media=1245.8s (20.76 min)  desvio=54.3s
  Random: media=1037.6s (17.29 min)  desvio=43.1s
  Steffen: media=913.1s (15.22 min)  desvio=25.1s


> Gráfico de barras comparando el tiempo promedio de cada política, separado por modalidad de vuelo.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

modalidades = ["corto", "largo"]
colores = ["tab:blue", "tab:orange", "tab:green", "tab:red"]

for ax, modalidad in zip(axes, modalidades):

    politicas_bonus = list(resultados_bonus[modalidad].keys())

    medias = [
        resultados_bonus[modalidad][pol][0] / 60
        for pol in politicas_bonus
    ]

    desvios = [
        resultados_bonus[modalidad][pol][1] / 60
        for pol in politicas_bonus
    ]

    ax.bar(
        politicas_bonus,
        medias,
        yerr=desvios,
        capsize=4,
        color=colores
    )

    ax.set_title(f"Modalidad: {modalidad}")
    ax.set_xlabel("Política")
    ax.tick_params(axis="x", rotation=20)
    for label in ax.get_xticklabels():
        label.set_ha("right")
    ax.grid(axis="y", alpha=0.3)

axes[0].set_ylabel("Tiempo promedio (minutos)")

fig.suptitle("Premium-Prioridad vs políticas originales (± desvío estándar)")
plt.tight_layout()
plt.show()

**Visualización animada del bonus**

Igual que con las 4 políticas originales, generamos la animación HTML autocontenida para la política **Premium-Prioridad**, en las 4 combinaciones de modalidad de vuelo (corto/largo) y orden intra-grupo (random/steffen). Esta versión (`crear_html_bonus`, copia de `crear_html` sin modificar la original) además:

- pinta cada asiento según la clase/grupo de embarque asignado (Premium = dorado, Grupo 1 = azul, Grupo 2 = turquesa, Grupo 3 = gris)
- dibuja a los pasajeros con discapacidad con forma cuadrada en vez de circular, para distinguirlos del resto

> Versión del HTML animado para el bonus: además pinta cada asiento según la clase y marca distinto a los pasajeros con discapacidad.

In [ ]:
# HTML AUTOCONTENIDO — VERSIÓN BONUS
#
# Copia de `crear_html` (no la reemplaza) que además:
#   - pinta cada asiento según la clase del pasajero asignado
#     (premium / grupo1 / grupo2 / grupo3)
#   - dibuja a los pasajeros con discapacidad con forma
#     cuadrada en vez de circular

def crear_html_bonus(historial, politica, archivo_salida):

    historial_json = json.dumps(
        historial,
        ensure_ascii=False
    )

    html = f"""
<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Boarding Simulation</title>

<style>

:root {{
    --bg: #07090d;
    --panel: #11151d;
    --seat: #171c26;
    --seat-border: #4d596a;

    --walking: #2f80ed;
    --stowing: #f2c94c;
    --interference: #eb5757;
    --sitting: #9b51e0;
    --seated: #27ae60;

    --text: #f5f7fb;
    --muted: #9aa4b2;
}}

* {{
    box-sizing: border-box;
}}

body {{
    margin: 0;
    background: var(--bg);
    color: var(--text);
    font-family: Arial, sans-serif;
}}

.app {{
    height: 100vh;
    display: grid;
    grid-template-columns: 1fr 260px;
    gap: 18px;
    padding: 18px;
}}

.stage {{
    position: relative;
    border: 1px solid #202734;
    border-radius: 18px;
    overflow: hidden;
    background: #090c12;
}}

.header {{
    position: absolute;
    top: 18px;
    left: 22px;
    z-index: 30;
}}

.header h1 {{
    margin: 0;
    font-size: 23px;
    letter-spacing: .08em;
}}

.header p {{
    margin: 6px 0 0;
    color: var(--muted);
    font-size: 13px;
}}

.policy {{
    display: inline-block;
    margin-top: 8px;
    padding: 5px 9px;
    border-radius: 999px;
    border: 1px solid #2d394c;
    background: #182131;
    font-size: 11px;
}}

.plane {{
    position: absolute;

    left: 52%;
    top: 54%;

    transform: translate(-50%, -50%);

    width: 420px;
    height: 88vh;

    max-height: 850px;
    min-height: 620px;

    border: 2px solid #697386;

    border-radius:
        48% 48% 18% 18%
        / 7% 7% 3% 3%;

    background: #0b0f16;
}}

.nose {{
    position: absolute;

    top: -28px;
    left: 50%;

    transform: translateX(-50%);

    width: 82px;
    height: 45px;

    border: 2px solid #697386;
    border-bottom: none;

    border-radius: 50% 50% 0 0;

    background: #0b0f16;
}}

.cabin {{
    position: absolute;

    inset:
        48px
        26px
        34px;

    display: grid;

    grid-template-rows:
        repeat(25, 1fr);

    gap: 2px;
}}

.row {{
    position: relative;

    display: grid;

    grid-template-columns:
        1fr
        1fr
        24px
        1fr
        1fr;

    align-items: center;

    gap: 5px;
}}

.row-number {{
    position: absolute;

    left: -22px;

    width: 16px;

    text-align: right;

    font-size: 8px;

    color: #758092;
}}

.seat {{
    height: 100%;
    max-height: 20px;
    min-height: 12px;

    border-radius: 4px;

    border:
        1px solid
        var(--seat-border);

    background:
        var(--seat);
}}

.aisle {{
    height: 100%;

    border-left:
        1px dashed #283141;

    border-right:
        1px dashed #283141;
}}

.person {{
    position: absolute;

    width: 12px;
    height: 12px;

    margin-left: -6px;
    margin-top: -6px;

    border-radius: 50%;

    z-index: 20;

    transition:
        left 90ms linear,
        top 90ms linear,
        background-color 90ms linear,
        transform 90ms linear;

    box-shadow:
        0 0 0 2px
        rgba(255,255,255,.12);
}}

.person.caminando {{
    background:
        var(--walking);
}}

.person.guardando {{
    background:
        var(--stowing);

    transform:
        scale(1.15);
}}

.person.interferencia {{
    background:
        var(--interference);

    transform:
        scale(1.2);
}}

.person.sentandose {{
    background:
        var(--sitting);
}}

.person.sentado {{
    background:
        var(--seated);
}}

/*
Pasajeros con discapacidad: misma paleta de
colores por estado, pero forma cuadrada en
vez de circular para distinguirlos.
*/
.person.discapacidad {{
    border-radius: 4px;

    width: 13px;
    height: 13px;

    margin-left: -6.5px;
    margin-top: -6.5px;

    box-shadow:
        0 0 0 2px
        rgba(255,255,255,.55);
}}

/*
Color de fondo de los asientos según la
clase / grupo de embarque del pasajero
asignado a ese asiento.
*/
.seat.clase-premium {{
    background: rgba(202, 161, 77, .35);
    border-color: #caa14d;
}}

.seat.clase-grupo1 {{
    background: rgba(91, 141, 239, .32);
    border-color: #5b8def;
}}

.seat.clase-grupo2 {{
    background: rgba(86, 194, 192, .32);
    border-color: #56c2c0;
}}

.seat.clase-grupo3 {{
    background: rgba(138, 143, 152, .32);
    border-color: #8a8f98;
}}

.bag {{
    position: absolute;

    width: 6px;
    height: 6px;

    right: -5px;
    bottom: -3px;

    border-radius: 1px;

    background: white;

    border: 1px solid #10131a;
}}

.queue {{
    position: absolute;

    left: 32px;
    bottom: 35px;

    width: 110px;
    height: 320px;
}}

.queue-title {{
    font-size: 10px;

    color: var(--muted);

    letter-spacing: .1em;
}}

.queue-person {{
    position: absolute;

    left: 45px;

    width: 10px;
    height: 10px;

    border-radius: 50%;

    background: #596273;
}}

.panel {{
    border-radius: 18px;

    border: 1px solid #202734;

    background: var(--panel);

    padding: 18px;

    display: flex;

    flex-direction: column;

    gap: 14px;
}}

.metric {{
    background: #0d1118;

    border: 1px solid #202734;

    border-radius: 12px;

    padding: 14px;
}}

.metric-label {{
    color: var(--muted);

    font-size: 10px;

    letter-spacing: .12em;

    text-transform: uppercase;
}}

.metric-value {{
    margin-top: 5px;

    font-size: 25px;

    font-weight: bold;
}}

.progress {{
    margin-top: 10px;

    height: 8px;

    background: #1b2230;

    border-radius: 999px;

    overflow: hidden;
}}

.progress-fill {{
    height: 100%;

    width: 0%;

    background:
        linear-gradient(
            90deg,
            #2f80ed,
            #27ae60
        );
}}

.legend {{
    display: grid;

    gap: 8px;

    margin-top: 9px;

    font-size: 12px;
}}

.legend-line {{
    display: flex;

    align-items: center;

    gap: 8px;
}}

.legend-dot {{
    width: 10px;
    height: 10px;

    border-radius: 50%;
}}

.controls {{
    margin-top: auto;

    display: grid;

    gap: 9px;
}}

button,
select {{

    width: 100%;

    border: 0;

    border-radius: 9px;

    padding: 11px;

    color: white;

    background: #1d2634;

    cursor: pointer;
}}

button.primary {{
    background: #2563eb;
}}

.big-clock {{
    position: absolute;

    right: 28px;
    top: 22px;

    z-index: 50;

    font-size: 36px;

    font-weight: bold;

    font-variant-numeric:
        tabular-nums;

    color: white;

    background:
        rgba(0,0,0,.40);

    border:
        1px solid
        #273244;

    padding:
        8px 14px;

    border-radius:
        12px;
}}

</style>

</head>

<body>

<div class="app">

<div class="stage">

    <div class="header">

        <h1>
        BOARDING SIMULATION
        </h1>

        <p>
        100 pasajeros ·
        25 filas ·
        4 asientos
        </p>

        <div class="policy">
        {politica}
        </div>

    </div>


    <div
        class="big-clock"
        id="bigClock"
    >
        0 s
    </div>


    <div
        class="queue"
        id="queue"
    >

        <div class="queue-title">
        PRÓXIMOS
        </div>

    </div>


    <div
        class="plane"
        id="plane"
    >

        <div class="nose">
        </div>

        <div
            class="cabin"
            id="cabin"
        >
        </div>

    </div>

</div>


<div class="panel">

    <div class="metric">

        <div class="metric-label">
        Segundo simulado
        </div>

        <div
            class="metric-value"
            id="timeValue"
        >
        0 s
        </div>

    </div>


    <div class="metric">

        <div class="metric-label">
        Sentados
        </div>

        <div class="metric-value">

            <span id="seatedValue">
            0
            </span>

            / 100

        </div>

        <div class="progress">

            <div
                class="progress-fill"
                id="progressBar"
            >
            </div>

        </div>

    </div>


    <div class="metric">

        <div class="metric-label">
        En el pasillo
        </div>

        <div
            class="metric-value"
            id="aisleValue"
        >
        0
        </div>

    </div>


    <div class="metric">

        <div class="metric-label">
        Esperando
        </div>

        <div
            class="metric-value"
            id="waitingValue"
        >
        100
        </div>

    </div>


    <div class="metric">

        <div class="metric-label">
        Estados
        </div>

        <div class="legend">

            <div class="legend-line">
                <span
                    class="legend-dot"
                    style="background:#2f80ed"
                ></span>
                Caminando
            </div>

            <div class="legend-line">
                <span
                    class="legend-dot"
                    style="background:#f2c94c"
                ></span>
                Guardando carry-on
            </div>

            <div class="legend-line">
                <span
                    class="legend-dot"
                    style="background:#eb5757"
                ></span>
                Interferencia
            </div>

            <div class="legend-line">
                <span
                    class="legend-dot"
                    style="background:#9b51e0"
                ></span>
                Sentándose
            </div>

            <div class="legend-line">
                <span
                    class="legend-dot"
                    style="background:#27ae60"
                ></span>
                Sentado
            </div>

        </div>

    </div>


    <div class="metric">

        <div class="metric-label">
        Clase / Grupo (color de asiento)
        </div>

        <div class="legend">

            <div class="legend-line">
                <span
                    class="legend-dot"
                    style="background:#caa14d"
                ></span>
                Premium (filas 1-5)
            </div>

            <div class="legend-line">
                <span
                    class="legend-dot"
                    style="background:#5b8def"
                ></span>
                Grupo 1
            </div>

            <div class="legend-line">
                <span
                    class="legend-dot"
                    style="background:#56c2c0"
                ></span>
                Grupo 2
            </div>

            <div class="legend-line">
                <span
                    class="legend-dot"
                    style="background:#8a8f98"
                ></span>
                Grupo 3
            </div>

            <div class="legend-line">
                <span
                    class="legend-dot"
                    style="background:#596273; border-radius:4px;"
                ></span>
                Con discapacidad (forma cuadrada)
            </div>

        </div>

    </div>


    <div class="controls">

        <select id="speedSelect">

            <option value="250">
            Lento
            </option>

            <option
                value="100"
                selected
            >
            Normal
            </option>

            <option value="45">
            Rápido
            </option>

        </select>

        <button
            class="primary"
            id="playBtn"
        >
        ▶ Reproducir
        </button>

        <button id="resetBtn">
        ↻ Reiniciar
        </button>

    </div>

</div>

</div>


<script>

/*
El historial real viene de Python.
No hay una segunda simulación en JavaScript.
*/

const historial =
{historial_json};


const cabin =
document.getElementById(
    "cabin"
);

const plane =
document.getElementById(
    "plane"
);

const queue =
document.getElementById(
    "queue"
);

const timeValue =
document.getElementById(
    "timeValue"
);

const bigClock =
document.getElementById(
    "bigClock"
);

const seatedValue =
document.getElementById(
    "seatedValue"
);

const aisleValue =
document.getElementById(
    "aisleValue"
);

const waitingValue =
document.getElementById(
    "waitingValue"
);

const progressBar =
document.getElementById(
    "progressBar"
);

const playBtn =
document.getElementById(
    "playBtn"
);

const resetBtn =
document.getElementById(
    "resetBtn"
);

const speedSelect =
document.getElementById(
    "speedSelect"
);


const seatOrder =
[-2, -1, 1, 2];


let frameActual = 0;

let reproduciendo = false;

let timer = null;


for (
    let fila = 1;
    fila <= 25;
    fila++
) {{

    const row =
    document.createElement(
        "div"
    );

    row.className =
    "row";

    row.dataset.row =
    fila;


    const numero =
    document.createElement(
        "span"
    );

    numero.className =
    "row-number";

    numero.textContent =
    fila;

    row.appendChild(
        numero
    );


    [-2, -1].forEach(
        asiento => {{

            const seat =
            document.createElement(
                "div"
            );

            seat.className =
            "seat";

            seat.dataset.row =
            fila;

            seat.dataset.seat =
            asiento;

            row.appendChild(
                seat
            );
        }}
    );


    const aisle =
    document.createElement(
        "div"
    );

    aisle.className =
    "aisle";

    row.appendChild(
        aisle
    );


    [1, 2].forEach(
        asiento => {{

            const seat =
            document.createElement(
                "div"
            );

            seat.className =
            "seat";

            seat.dataset.row =
            fila;

            seat.dataset.seat =
            asiento;

            row.appendChild(
                seat
            );
        }}
    );


    cabin.appendChild(
        row
    );
}}


/*
Coloreamos cada asiento según la clase / grupo
del pasajero que le fue asignado. La clase de
cada asiento no cambia durante la simulación,
así que alcanza con mirar el primer frame.
*/

const claseDeAsiento = {{}};

for (
    const p
    of historial[0].pasajeros
) {{

    claseDeAsiento[
        `${{p.fila}}-${{p.asiento}}`
    ] = p.clase;
}}

const claseCss = {{
    "primera_clase": "clase-premium",
    "grupo1": "clase-grupo1",
    "grupo2": "clase-grupo2",
    "grupo3": "clase-grupo3",
}};

document
.querySelectorAll(".seat")
.forEach(
    seat => {{

        const clave =
        `${{seat.dataset.row}}-${{seat.dataset.seat}}`;

        const clase =
        claseDeAsiento[clave];

        const cssClass =
        claseCss[clase];

        if (cssClass) {{
            seat.classList.add(cssClass);
        }}
    }}
);


const nombres =
historial[0]
.pasajeros
.map(
    p => p.nombre
);


for (
    const nombre
    of nombres
) {{

    const persona =
    document.createElement(
        "div"
    );

    persona.className =
    "person";

    persona.id =
    `persona-${{nombre}}`;

    persona.style.display =
    "none";

    plane.appendChild(
        persona
    );
}}


function posicionPasillo(
    fila
) {{

    const row =
    cabin.querySelector(
        `.row[data-row="${{fila}}"]`
    );

    const rowRect =
    row.getBoundingClientRect();

    const planeRect =
    plane.getBoundingClientRect();

    return {{

        left:
        plane.clientWidth / 2,

        top:
        rowRect.top
        - planeRect.top
        + rowRect.height / 2

    }};
}}


function posicionAsiento(
    fila,
    asiento
) {{

    const seat =
    cabin.querySelector(
        `.seat[data-row="${{fila}}"][data-seat="${{asiento}}"]`
    );

    const rect =
    seat.getBoundingClientRect();

    const planeRect =
    plane.getBoundingClientRect();

    return {{

        left:
        rect.left
        - planeRect.left
        + rect.width / 2,

        top:
        rect.top
        - planeRect.top
        + rect.height / 2

    }};
}}


function actualizarCola(
    pasajeros
) {{

    queue
    .querySelectorAll(
        ".queue-person"
    )
    .forEach(
        el => el.remove()
    );


    pasajeros
    .filter(
        p =>
        p.estado ===
        "esperando"
    )
    .slice(
        0,
        14
    )
    .forEach(
        (p, indice) => {{

            const el =
            document.createElement(
                "div"
            );

            el.className =
            "queue-person";

            el.style.top =
            `${{28 + indice * 19}}px`;


            if (
                p.carry_on
            ) {{

                const bag =
                document.createElement(
                    "div"
                );

                bag.className =
                "bag";

                el.appendChild(
                    bag
                );
            }}


            queue.appendChild(
                el
            );
        }}
    );
}}


function actualizarPasajero(
    p
) {{

    const el =
    document.getElementById(
        `persona-${{p.nombre}}`
    );


    el.className =
    `person ${{p.estado}}` +
    (
        p.tipo === "discapacidad"
        ? " discapacidad"
        : ""
    );


    if (
        p.estado ===
        "esperando"
    ) {{

        el.style.display =
        "none";

        return;
    }}


    el.style.display =
    "block";


    /*
    Creamos o quitamos la
    mini valija.
    */

    let bag =
    el.querySelector(
        ".bag"
    );


    if (
        p.carry_on
        && !bag
    ) {{

        bag =
        document.createElement(
            "div"
        );

        bag.className =
        "bag";

        el.appendChild(
            bag
        );
    }}


    if (
        !p.carry_on
        && bag
    ) {{

        bag.remove();
    }}


    let pos;


    if (
        p.estado ===
        "sentado"
    ) {{

        pos =
        posicionAsiento(
            p.fila,
            p.asiento
        );

    }}

    else if (
        p.estado ===
        "sentandose"
        && p.posicion === null
    ) {{

        /*
        Este pasajero ya estaba sentado,
        se levantó por una interferencia
        y ahora está volviendo a sentarse.
        */

        pos =
        posicionAsiento(
            p.fila,
            p.asiento
        );

    }}

    else if (
        p.estado ===
        "interferencia"
        && p.posicion === null
    ) {{

        /*
        Este pasajero ya estaba sentado
        y tuvo que levantarse para dejar pasar.
        */

        pos =
        posicionAsiento(
            p.fila,
            p.asiento
        );

    }}

    else if (
        p.posicion !== null
        && p.posicion > 0
    ) {{

        pos =
        posicionPasillo(
            p.posicion
        );

    }}

    else {{

        return;
    }}


    el.style.left =
    `${{pos.left}}px`;

    el.style.top =
    `${{pos.top}}px`;
}}


function mostrarFrame(
    indice
) {{

    const frame =
    historial[indice];


    const pasajeros =
    frame.pasajeros;


    /*
    El segundo mostrado es
    exactamente el segundo
    guardado por Python.
    */

    timeValue.textContent =
    `${{frame.segundo}} s`;

    bigClock.textContent =
    `${{frame.segundo}} s`;


    let sentados = 0;

    let esperando = 0;

    let pasillo = 0;


    for (
        const p
        of pasajeros
    ) {{

        actualizarPasajero(
            p
        );


        if (
            p.estado ===
            "sentado"
        ) {{
            sentados++;
        }}


        if (
            p.estado ===
            "esperando"
        ) {{
            esperando++;
        }}


        if (
            [
                "caminando",
                "guardando",
                "interferencia",
                "sentandose"
            ].includes(
                p.estado
            )
            &&
            p.posicion !== null
        ) {{
            pasillo++;
        }}
    }}


    seatedValue.textContent =
    sentados;

    waitingValue.textContent =
    esperando;

    aisleValue.textContent =
    pasillo;

    progressBar.style.width =
    `${{sentados}}%`;


    actualizarCola(
        pasajeros
    );
}}


function avanzar() {{

    if (
        frameActual
        >= historial.length
    ) {{

        pausar();

        return;
    }}


    mostrarFrame(
        frameActual
    );


    frameActual++;
}}


function reproducir() {{

    if (
        reproduciendo
    ) {{
        return;
    }}


    reproduciendo =
    true;


    playBtn.textContent =
    "⏸ Pausar";


    const velocidad =
    Number(
        speedSelect.value
    );


    timer =
    setInterval(
        avanzar,
        velocidad
    );
}}


function pausar() {{

    reproduciendo =
    false;


    clearInterval(
        timer
    );


    timer =
    null;


    playBtn.textContent =
    "▶ Reproducir";
}}


function reiniciar() {{

    pausar();


    frameActual =
    0;


    mostrarFrame(
        0
    );
}}


playBtn
.addEventListener(
    "click",
    () => {{

        if (
            reproduciendo
        ) {{
            pausar();
        }}

        else {{
            reproducir();
        }}
    }}
);


resetBtn
.addEventListener(
    "click",
    reiniciar
);


speedSelect
.addEventListener(
    "change",
    () => {{

        if (
            reproduciendo
        ) {{

            pausar();

            reproducir();
        }}
    }}
);


window
.addEventListener(
    "resize",
    () => {{

        mostrarFrame(
            Math.min(
                frameActual,
                historial.length - 1
            )
        );
    }}
);


mostrarFrame(
    0
);

</script>

</body>
</html>
"""

    Path(
        archivo_salida
    ).write_text(
        html,
        encoding="utf-8"
    )

> Corremos Premium-Prioridad en las dos modalidades de vuelo y con las dos variantes de orden interno, generando sus animaciones.

In [ ]:
# EJECUTAR "PREMIUM-PRIORIDAD" EN LAS DOS MODALIDADES,
# CON LAS DOS VARIANTES DE ORDEN INTRA-GRUPO (random / steffen)
#
# Reutiliza simular_bonus (definida en la celda anterior) y
# crear_html_bonus (definida arriba en esta misma celda), sin
# tocar crear_html ni el loop de las 4 políticas originales.

for modalidad in ["corto", "largo"]:

    for orden_intragrupo in ["random", "steffen"]:

        tiempo_total, historial = simular_bonus(modalidad, orden_intragrupo)

        print("\n-----------------------------")
        print(f"Política: Premium-Prioridad — {modalidad} / intra-grupo {orden_intragrupo}")
        print("-----------------------------")
        print("Tiempo total:", tiempo_total, "segundos")
        print("Tiempo total:", round(tiempo_total / 60, 2), "minutos")

        archivo_html = f"boarding_premium_prioridad_{modalidad}_{orden_intragrupo}_animado.html"

        crear_html_bonus(
            historial,
            f"Premium-Prioridad ({modalidad}, intra-grupo {orden_intragrupo})",
            archivo_html
        )

        print("HTML creado:", archivo_html)

print("\nSe generaron las animaciones del bonus.")


-----------------------------
Política: Premium-Prioridad — corto / intra-grupo random
-----------------------------
Tiempo total: 1584 segundos
Tiempo total: 26.4 minutos
HTML creado: boarding_premium_prioridad_corto_random_animado.html

-----------------------------
Política: Premium-Prioridad — corto / intra-grupo steffen
-----------------------------
Tiempo total: 1394 segundos
Tiempo total: 23.23 minutos
HTML creado: boarding_premium_prioridad_corto_steffen_animado.html

-----------------------------
Política: Premium-Prioridad — largo / intra-grupo random
-----------------------------
Tiempo total: 1339 segundos
Tiempo total: 22.32 minutos
HTML creado: boarding_premium_prioridad_largo_random_animado.html

-----------------------------
Política: Premium-Prioridad — largo / intra-grupo steffen
-----------------------------
Tiempo total: 1241 segundos
Tiempo total: 20.68 minutos
HTML creado: boarding_premium_prioridad_largo_steffen_animado.html

Se generaron las animaciones del bon